In [0]:
from pyspark.sql import functions as F

# Config
catalog, schema = "dlt_dbx", "default"
base       = f"/Volumes/{catalog}/{schema}/healthcare_data"
source_dir = f"{base}/patients_daily/"
schema_loc = f"{base}/_schemas/patients_daily"
chk_loc    = f"{base}/_checkpoints/patients_daily"
target     = f"{catalog}.{schema}.raw_patients_daily"

# Read: Auto Loader picks up only new files
raw = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_loc)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaHints", "admission_date DATE")
    .option("pathGlobFilter", "patients_daily_file_*.csv")
    .load(source_dir)
)

# Enrich: lineage columns for traceability and replay
enriched = (
    raw
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("file_modified_at", F.col("_metadata.file_modification_time"))
    .withColumn("ingested_at", F.current_timestamp())
)

# Write: append-only, no drops, runs and stops (event-driven)
query = (
    enriched.writeStream
    .option("checkpointLocation", chk_loc)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target)
)

query.awaitTermination()
print(f"Ingested {query.lastProgress['numInputRows'] if query.lastProgress else 0} rows into {target}")